# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row per content item (page), observed over a trailing 90-day window ending at export time.

**Time window:** all metrics span the most recent 90 days prior to export. This is a trailing window, not a fixed calendar period.

In [ ]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('Total rows:', len(df))
print('Grain check (should be 30000 unique content_ids with 1 row each):')
grain = df.groupby('content_id').size().reset_index(name='count')
print('Rows per content_id:', grain['count'].unique())
print('Grain holds (all 1 row per content_id):', grain['count'].nunique() == 1)

print('Date range check - all rows in trailing 90d window:')
print('All content_ids have data:', df['content_id'].nunique() == 30000)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
`content_id` | Context | Pseudonymous ID for grouping/joins only; never a model feature |
`client_id` | Context | Used for client-holdout splits and grouping; pseudonym |
`search_volume` | Feature | Keyword search volume, knowable before prediction |
`competition` | Feature | Keyword competition score 0–1, knowable before prediction |
`competition_level` | Feature | Categorical: LOW/MEDIUM/HIGH, knowable before prediction |
`cpc` | Feature | Cost-per-click estimate, knowable before prediction |
`content_type` | Feature | Categorical: keyword/feedly/comparison article, knowable before prediction |
`main_intent` | Feature | Categorical: informational/transactional/commercial/navigational, knowable before prediction |
`word_count` | Feature | Article length; has systematic missingness per content_type |
`char_count` | Feature | Character count; missing alongside word_count |
`provider_used` | Context | LLM provider for content generation; grouping/splitting only |
`model_used` | Context | LLM model name; grouping/splitting only |
`impressions_90d` | Feature | GSC impressions over 90d; always ≥ 1 in this slice, knowable before prediction |
`clicks_90d` | Feature | GSC clicks over 90d, knowable before prediction |
`pageviews_90d` | Feature | GA4 pageviews over 90d, knowable before prediction |
`sessions_90d` | Feature | GA4 sessions over 90d, knowable before prediction |
`users_90d` | Feature | GA4 users over 90d, knowable before prediction |
`engaged_sessions_90d` | Feature | GA4 engaged sessions over 90d, knowable before prediction |
`ai_sessions_90d` | Feature | AI-referred sessions over 90d; can exceed 100%, knowable before prediction |
`scroll_events_90d` | Feature | GA4 scroll events; can exceed 100% (multiple per pageview) |
`days_with_impressions` | Feature | Days with ≥ 1 impression in 90d window (0–90) |
`days_with_sessions` | Feature | Days with ≥ 1 session in 90d window (0–90) |
`impressions_last_30d` | Feature | Last 30d impressions; align window carefully with label |
`clicks_last_30d` | Feature | Last 30d clicks |
`sessions_last_30d` | Feature | Last 30d sessions |
`impressions_prev_30d` | Feature | Days 31–60 back impressions; safe if label uses last-30d window |
`clicks_prev_30d` | Feature | Clicks days 31–60 back |
`sessions_prev_30d` | Feature | Sessions days 31–60 back |
`content_age_days` | Feature | Days since content creation; every row ≥ 90 in this slice |
`age_tier` | Feature | Bucketized from content_age_days; transparency bucket |
`age_tier_order` | Feature | Numeric order of age_tier (1 = youngest) |
`days_since_last_update` | Feature | Days since last content update |
`freshness_tier` | Feature | Bucketized from days_since_last_update; never/0-30/31-90/91-180/181+ |
`word_count_tier` | Feature | Bucketized from word_count; blank when word_count is blank |
`char_count_tier` | Feature | Bucketized from char_count; blank when char_count is blank |
`ctr` | Feature | Click-through rate × 100%; `0.76` = 0.76% (rate column ×100) |
`avg_position` | Feature | Mean GSC position; `0` = no data (not rank zero; 1,205 rows) |
`engagement_rate` | Feature | Engaged sessions / sessions × 100%; knowable before prediction |
`scroll_rate` | Feature | Scroll events / pageviews × 100%; **can exceed 100%** |
`ai_traffic_pct` | Feature | AI sessions / sessions × 100%; **can exceed 100%** |
`impression_tier` | Feature | Bucketized impression volume; transparent thresholds |
`position_tier` | Feature | Bucketized position; no_data/top_3/page_1/striking/page_3_5/deep |
`trend_direction` | **Label source** | Computed from trend_pct; must **never** be a feature — label derivation |
`trend_pct` | **Label source** | `(last30d − prev30d) / prev30d × 100`; blank when prev30d=0 (3,388 rows); **must never** be a feature |
`is_declining_label` | **Label / proxy** | Target: 1 when trend_direction == "down" (54.2% of rows); this is the prediction target

In [ ]:
print()
print('--- Missingness per major column, overall and by content_type ---')
for col in ['search_volume', 'competition', 'word_count', 'provider_used', 'model_used', 'content_type']:
    overall = df[col].isna().mean()
    by_type = df.groupby('content_type')[col].apply(lambda x: x.isna().mean()).round(3)
    print(f'{col:20s} overall={overall:.3f}')
    print(by_type)
    print()

In [ ]:
print()
print('--- Missingness overall ---')
for col in ['search_volume', 'competition', 'word_count', 'provider_used', 'model_used']:
    overall = df[col].isna().mean()
    print(f'{col:20s} missing={overall:.3f}')
print()
print('--- Missingness by content_type ---')
for col in ['search_volume', 'word_count']:
    by_type = df.groupby('content_type')[col].apply(lambda x: x.isna().mean()).round(3)
    print(f'{col:20s}:')
    print(by_type)
print()
print('--- Verifying trailing 90d window ---')
print(f'impressions_90d min: {df["impressions_90d"].min()} (expected ≥ 1)')
print(f'impressions_90d max: {df["impressions_90d"].max()}')
print(f'rows with impressions_90d = 0: {(df["impressions_90d"] == 0).sum()} (expected 0 in this slice)')

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Grain verification:**

```sql
-- Should return 30000 rows, one per content_id
SELECT content_id, COUNT(*) c FROM content_refresh_anonymized GROUP BY content_id HAVING c > 1 LIMIT 5
-- Output: empty (grain holds)
```

**Counts check:**

```sql
-- Total rows
SELECT COUNT(*) FROM content_refresh_anonymized
-- Rows per content_type
SELECT content_type, COUNT(*) FROM content_refresh_anonymized GROUP BY content_type
```

**Missingness check:**

```sql
-- Overall missing fraction per column
SELECT COUNT(CASE WHEN search_volume IS NULL THEN 1 END) * 1.0 / COUNT(*) AS search_volume_missing
FROM content_refresh_anonymized
-- Missingness by content_type
SELECT content_type,
       COUNT(CASE WHEN word_count IS NULL THEN 1 END) * 1.0 / COUNT(*) AS word_count_missing
FROM content_refresh_anonymized
GROUP BY content_type
```

**Window check:**

```sql
-- Date range per content_type (to verify trailing 90d window)
SELECT content_type, MIN(report_date), MAX(report_date)
FROM fact_content_daily_performance
GROUP BY content_type
-- (warehouse table; use to verify window alignment)
```

In [ ]:
print()
print('--- Grain verification ---')
grain = df.groupby('content_id').size().reset_index(name='count')
print('Rows per content_id unique values:', grain['count'].unique())
print('Grain holds (all 1 row per content_id):', grain['count'].nunique() == 1)

print('--- Counts ---')
print('Total rows:', len(df))
print('Rows per content_type:')
print(df['content_type'].value_counts())

print('--- Missingness ---')
for col in ['search_volume', 'word_count']:
    overall = df[col].isna().mean()
    print(f'{col:20s} overall missing fraction: {overall:.3f}')
    by_type = df.groupby('content_type')[col].apply(lambda x: x.isna().mean()).round(3)
    print(by_type)
print()
print('--- Trailing 90d window verification ---')
print(f'impressions_90d min: {df["impressions_90d"].min()} (expected ≥ 1)')
print(f'impressions_90d max: {df["impressions_90d"].max()}')
print(f'rows with impressions_90d = 0: {(df["impressions_90d"] == 0).sum()} (expected 0 in this slice)')

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced panel:** Per-client history depth varies wildly (some clients have 17 months, some as little as 3). `dim_clients.gsc_data_start` marks the per-client start date — earlier rows have GSC data but may lack GA4.
- **GSC-only early rows:** Before each client's `ga4_data_start`, GA4 columns (sessions, users, etc.) are zero-FILLED with `ga4_data_available = FALSE`. Zeros there are not 'no engagement'; they are placeholder values.
- **Window overlap with label:** The `*_last_30d` columns overlap the label's 30-day window. If predicting decline over the last 30 days, only `*_prev_30d` columns are safe features; the last-30d columns constitute forward information.
- **Rate ×100 gotcha:** `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct` are all ×100 percentages. `ctr = 0.76` means 0.76%, not 76%. A blind `fillna(0)` on these injects a category signal.
- **Systematic missingness:** `search_volume`, `competition`, `cpc` are blank for rows with no keyword data (2,468 rows). `word_count` blank for 7,699 rows. Missingness follows `content_type` — a blind `fillna(0)` encodes content type.
- **`avg_position = 0`:** Means 'no position data', not rank zero. 1,205 rows have `avg_position = 0`.